In [2]:
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())

True

In [3]:
from openai import OpenAI
openai_client  = OpenAI()

In [3]:
response = openai_client.responses.create(
    model="gpt-4o-mini",
    input="Write a short bedtime story about a unicorn"
)
print(response.output_text)

**The Whispering Woods and the Unicorn**

Once upon a time, in a hidden valley called the Whispering Woods, there lived a beautiful unicorn named Luna. Her coat shimmered like silver under the moonlight, and her horn glowed like a star. The animals of the woods adored her, for she had a special gift: she could bring sweet dreams to those who believed.

Every evening, as the sun dipped below the horizon, Luna would trot through the woods, sprinkling a magical dust that danced in the air like tiny fireflies. The trees would lean closer to listen as she whispered enchanting stories of hope and adventure to the creatures nestled in the underbrush.

One night, as Luna wandered near the edge of the woods, she heard a soft sobbing sound. Curious, she followed it to find a little girl named Mia sitting on a rock, tears glistening in her eyes. "What troubles you, dear one?" Luna asked gently.

"I can’t sleep because I’m afraid of the dark," Mia replied, her voice quavering.

With a kind smile, 

In [4]:
messages = [
    {"role": "user", "content": "tell me a joke about Alexey"}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages
)


In [5]:
print(response.model_dump_json(indent=2))


{
  "id": "resp_048ee8445e89feac0069e03e8543a48196a3c7edf9e44f2fd3",
  "created_at": 1776303749.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "gpt-4o-mini-2024-07-18",
  "object": "response",
  "output": [
    {
      "id": "msg_048ee8445e89feac0069e03e864e0c8196afe692f1df26d036",
      "content": [
        {
          "annotations": [],
          "text": "Why did Alexey bring a ladder to the bar?\n\nBecause he heard the drinks were on the house!",
          "type": "output_text",
          "logprobs": []
        }
      ],
      "role": "assistant",
      "status": "completed",
      "type": "message",
      "phase": null
    }
  ],
  "parallel_tool_calls": true,
  "temperature": 1.0,
  "tool_choice": "auto",
  "tools": [],
  "top_p": 1.0,
  "background": false,
  "completed_at": 1776303750.0,
  "conversation": null,
  "max_output_tokens": null,
  "max_tool_calls": null,
  "previous_response_id": null,
  "prompt": null,
  "promp

In [6]:


chat_history = [
    {"role": "system", "content": "You are a helpful assistant."}
]

def chat_with_history(user_text: str) -> str:
    chat_history.append({"role": "user", "content": user_text})

    response = openai_client.responses.create(
        model="gpt-4o-mini",
        input=chat_history
    )

    assistant_text = response.output_text
    chat_history.append({"role": "assistant", "content": assistant_text})
    return assistant_text

# Example
print(chat_with_history("Hello! Can you introduce yourself?"))
print(chat_with_history("Please ask me a follow-up question."))

Hello! I'm an AI assistant here to help you with a wide range of questions and tasks. Whether you need information, recommendations, or assistance with problem-solving, feel free to ask! How can I assist you today?
Sure! What topic or area are you interested in discussing today?


In [7]:
print(chat_with_history("breif me the difference between langchain and pydantic?"))

Certainly! Here’s a brief overview of the key differences between LangChain and Pydantic:

### LangChain
- **Purpose**: LangChain is a framework designed for building applications that leverage language models (like GPT).
- **Features**: It includes tools for chaining together different components (like prompt templates, memory, and agents), making it easier to create complex workflows with language models.
- **Use Cases**: Ideal for developing chatbots, virtual assistants, and applications that require integrating various language model capabilities.

### Pydantic
- **Purpose**: Pydantic is a data validation and settings management library for Python.
- **Features**: It allows you to define data models with type annotations, automatically validating data and converting it into the specified types. It can be used to ensure data integrity and structure.
- **Use Cases**: Commonly used in applications where data validation is critical, such as API responses or configurations in web framew

In [4]:
## testing the chunking document code
from docx import Document

In [14]:
from typing import Any, Dict, Iterable, List


def sliding_window(
    seq: Iterable[Any],
    size: int,
    step: int,
) -> List[Dict[str, Any]]:
    """Create overlapping chunks from a sequence using a sliding window approach.

    Args:
        seq: The input sequence (string or list) to be chunked.
        size: The size of each chunk/window.
        step: The step size between consecutive windows.

    Returns:
        A list of dictionaries, each containing:
            - 'start': The starting position of the chunk in the original sequence
            - 'content': The chunk content

    Raises:
        ValueError: If size or step are not positive integers.

    Example:
        >>> sliding_window("hello world", size=5, step=3)
        [{'start': 0, 'content': 'hello'}, {'start': 3, 'content': 'lo wo'}]
    """
    if size <= 0 or step <= 0:
        raise ValueError("size and step must be positive")

    n = len(seq)
    result = []
    for i in range(0, n, step):
        batch = seq[i : i + size]
        result.append({"start": i, "content": batch})
        if i + size > n:
            break

    return result



def chunk_documents(
    documents: Iterable[Dict[str, str]],
    size: int = 2000,
    step: int = 1000,
    content_field_name: str = "content",
) -> List[Dict[str, str]]:
    """Split a collection of documents into smaller chunks using sliding windows.

    Takes documents and breaks their content into overlapping chunks while preserving
    all other document metadata (filename, etc.) in each chunk.

    Args:
        documents: An iterable of document dictionaries. Each document must have a content field.
        size: The maximum size of each chunk. Defaults to 2000.
        step: The step size between chunks. Defaults to 1000.
        content_field_name: The name of the field containing document content.
            Defaults to 'content'.

    Returns:
        A list of chunk dictionaries. Each chunk contains:
            - All original document fields except the content field
            - 'start': Starting position of the chunk in original content
            - 'content': The chunk content

    Example:
        >>> documents = [{'content': 'long text...', 'filename': 'doc.txt'}]
        >>> chunks = chunk_documents(documents, size=100, step=50)
        >>> # Or with custom content field:
        >>> documents = [{'text': 'long text...', 'filename': 'doc.txt'}]
        >>> chunks = chunk_documents(documents, content_field_name='text')
    """
    results = []

    for doc in documents:
        doc_copy = doc.copy()
        doc_content = doc_copy.pop(content_field_name)
        chunks = sliding_window(doc_content, size=size, step=step)
        for chunk in chunks:
            chunk.update(doc_copy)
        results.extend(chunks)

    return results



In [5]:
doc_path = "unicorn_story.docx"
doc = Document(doc_path)

In [11]:
for i in doc.paragraphs:
    print(i.text, end="\n\n")

Once upon a time, in a hidden valley called the Whispering Woods, there lived a beautiful unicorn named Luna. Her coat shimmered like silver under the moonlight, and her horn glowed like a star. The animals of the woods adored her, for she had a special gift: she could bring sweet dreams to those who believed.Every evening, as the sun dipped below the horizon, Luna would trot through the woods, sprinkling a magical dust that danced in the air like tiny fireflies. The trees would lean closer to listen as she whispered enchanting stories of hope and adventure to the creatures nestled in the underbrush.







In [12]:
full_text = "\n".join([para.text for para in doc.paragraphs if para.text.strip()])
print(f"Total characters: {len(full_text)}")
print(f"Preview:\n{full_text[:300]}")

Total characters: 607
Preview:
Once upon a time, in a hidden valley called the Whispering Woods, there lived a beautiful unicorn named Luna. Her coat shimmered like silver under the moonlight, and her horn glowed like a star. The animals of the woods adored her, for she had a special gift: she could bring sweet dreams to those wh


In [13]:
# Wrap into document format
documents = [{"content": full_text, "filename": "unicorn_story.docx"}]
print(f"Number of documents: {len(documents)}")
print(f"Keys in document: {list(documents[0].keys())}")

Number of documents: 1
Keys in document: ['content', 'filename']


In [15]:
# Use small size/step so you can see the overlap clearly on a short doc
chunks_raw = sliding_window(full_text, size=200, step=100)

print(f"Total chunks: {len(chunks_raw)}\n")
for i, chunk in enumerate(chunks_raw):
    print(f"--- Chunk {i} | start={chunk['start']} | len={len(chunk['content'])} ---")
    print(chunk['content'])
    print()

Total chunks: 6

--- Chunk 0 | start=0 | len=200 ---
Once upon a time, in a hidden valley called the Whispering Woods, there lived a beautiful unicorn named Luna. Her coat shimmered like silver under the moonlight, and her horn glowed like a star. The a

--- Chunk 1 | start=100 | len=200 ---
med Luna. Her coat shimmered like silver under the moonlight, and her horn glowed like a star. The animals of the woods adored her, for she had a special gift: she could bring sweet dreams to those wh

--- Chunk 2 | start=200 | len=200 ---
nimals of the woods adored her, for she had a special gift: she could bring sweet dreams to those who believed.Every evening, as the sun dipped below the horizon, Luna would trot through the woods, sp

--- Chunk 3 | start=300 | len=200 ---
o believed.Every evening, as the sun dipped below the horizon, Luna would trot through the woods, sprinkling a magical dust that danced in the air like tiny fireflies. The trees would lean closer to l

--- Chunk 4 | start=400 |

In [ ]:
#sliding window function Takes raw text (just a string)
#Returns chunks with only: {'start': 0, 'content': '...'}
#No metadata preserved — you lose filename, source, or any other document info
chunks_raw

[{'start': 0,
  'content': 'Once upon a time, in a hidden valley called the Whispering Woods, there lived a beautiful unicorn named Luna. Her coat shimmered like silver under the moonlight, and her horn glowed like a star. The a'},
 {'start': 100,
  'content': 'med Luna. Her coat shimmered like silver under the moonlight, and her horn glowed like a star. The animals of the woods adored her, for she had a special gift: she could bring sweet dreams to those wh'},
 {'start': 200,
  'content': 'nimals of the woods adored her, for she had a special gift: she could bring sweet dreams to those who believed.Every evening, as the sun dipped below the horizon, Luna would trot through the woods, sp'},
 {'start': 300,
  'content': 'o believed.Every evening, as the sun dipped below the horizon, Luna would trot through the woods, sprinkling a magical dust that danced in the air like tiny fireflies. The trees would lean closer to l'},
 {'start': 400,
  'content': 'rinkling a magical dust that danced 

In [18]:
#If you're chunking one document, sliding window works fine.
#If you're chunking multiple documents (e.g., 100 PDFs), chunk document function 
#is essential — otherwise you can't trace which chunk came from which file. '
#'You'd end up with 500 chunks and no idea

In [19]:
chunks = chunk_documents(documents, size=200, step=100)

print(f"Total chunks: {len(chunks)}\n")
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i} | start={chunk['start']} | filename={chunk['filename']} | len={len(chunk['content'])} ---")
    print(chunk['content'])
    print()

Total chunks: 6

--- Chunk 0 | start=0 | filename=unicorn_story.docx | len=200 ---
Once upon a time, in a hidden valley called the Whispering Woods, there lived a beautiful unicorn named Luna. Her coat shimmered like silver under the moonlight, and her horn glowed like a star. The a

--- Chunk 1 | start=100 | filename=unicorn_story.docx | len=200 ---
med Luna. Her coat shimmered like silver under the moonlight, and her horn glowed like a star. The animals of the woods adored her, for she had a special gift: she could bring sweet dreams to those wh

--- Chunk 2 | start=200 | filename=unicorn_story.docx | len=200 ---
nimals of the woods adored her, for she had a special gift: she could bring sweet dreams to those who believed.Every evening, as the sun dipped below the horizon, Luna would trot through the woods, sp

--- Chunk 3 | start=300 | filename=unicorn_story.docx | len=200 ---
o believed.Every evening, as the sun dipped below the horizon, Luna would trot through the woods, sprinklin

In [ ]:
## reall world example: 
# documents = [
#     {'content': 'text from file1...', 'filename': 'doc1.pdf', 'date': '2024-01-01'},
#     {'content': 'text from file2...', 'filename': 'doc2.pdf', 'date': '2024-02-15'}
# ]

# chunks = chunk_documents(documents, size=500, step=250)
# Every chunk now has filename + date attached for retrieval/citations